In [1]:
import sys
sys.path.append('../')
from preload_data import load_empathetic_data

train, val, test, vocab = load_empathetic_data('../data')

/home/new/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


加载数据: ../data/dataset_preproc.p
                                      Opts                                      
--------------------------------------------------------------------------------
                               data_dir: data/ED                                
                              emo_input: self_att                               
                            emo_combine: gate                                   
                                decoder: single                                 
                             hidden_dim: 300                                    
                                emb_dim: 300                                    
                             batch_size: 16                                     
                                     lr: 0.0001                                 
                          max_grad_norm: 2.0                                    
                              beam_size: 5                                   

In [2]:
import re

# Rename variable to avoid confusion with the dataset key 'target'
search_query = r"two fabolous weedings of my two fabulous"
print(f"Searching for: '{search_query}'")
data_dict = {
    'trian': train, 'val': val, 'test': test
}
find_flag = False

# Iterate over datasets
for loader_name, loader in zip(['train', 'val', 'test'], [train, val, test]):
    length = len(loader['context'])
    keylist = ['context', 'emotion', 'target', 'situation']
    
    print(f"Scanning {loader_name} set ({length} samples)...")
    
    for i in range(length):
        # RESET FLAG FOR EACH SAMPLE
        
        for key in keylist:
            reference = loader[key][i]
            combined = ''
            
            # LIST of strings (target, situation)
            if key in ['target', 'situation']:
                combined = ' '.join(reference)
                if re.search(search_query, combined, re.IGNORECASE):
                    find_flag = True
            
            # EMOTION (String/Numpy String)
            elif key == 'emotion':
                combined = str(reference)
                if re.search(search_query, combined, re.IGNORECASE):
                    find_flag = True

            # CONTEXT (List of List of strings)
            elif key == 'context':
                for inner_ctx in reference:
                    combined = ' '.join(inner_ctx)
                    if re.search(search_query, combined, re.IGNORECASE):
                        find_flag = True
                        break # Stop checking context turns if found
            
            if find_flag: break # Stop checking keys if found in this sample

        if find_flag:
            print(f'Match found in {loader_name} set, idx {i}')
            # sys.exit(0)
            break
    if find_flag: break


Searching for: 'two fabolous weedings of my two fabulous'
Scanning train set (40250 samples)...
Match found in train set, idx 22034


In [ ]:
# Retrieve the correct loader based on the found name
# Re-define mapping to ensure 'train' key is correct (fixing previous typo 'trian')
lname, pos = loader_name, i
loaders_map = {'train': train, 'val': val, 'test': test}
current_loader = loaders_map.get(lname)

if current_loader:
    print(f"🔎 INSPECTING: [{lname}] set, Index [{pos}]")
    print("=" * 80)
    
    # 1. Emotion (Scalar)
    emotion = current_loader['emotion'][pos]
    print(f"❤️  EMOTION: {emotion}\n")
    
    # 2. Situation (List of tokens)
    situation_tokens = current_loader['situation'][pos]
    situation_str = " ".join(situation_tokens)
    print(f"📝 SITUATION:\n    {situation_str}\n")
    
    # 3. Context (List of List of tokens)
    print(f"💬 CONTEXT (Dialogue History):")
    context_turns = current_loader['context'][pos]
    for idx, turn_tokens in enumerate(context_turns):
        turn_str = " ".join(turn_tokens)
        # Assuming alternating speakers, though dataset specific
        print(f"    [Turn {idx+1}]: {turn_str}")
        
    # 4. Target (List of tokens)
    target_tokens = current_loader['target'][pos]
    target_str = " ".join(target_tokens)
    print(f"\n🎯 TARGET RESPONSE:\n    {target_str}")
    
    print("=" * 80)
else:
    print(f"Error: Loader '{lname}' not found.")

In [3]:
train.keys()

dict_keys(['context', 'target', 'emotion', 'situation', 'emotion_context', 'utt_cs', 'comet_cxt', 'comet_sit', 'comet_trg'])

In [5]:
# Print samples in simple nested text format
def print_sample(data, index):
    print(f"{'='*60}")
    print(f"Sample #{index}")
    print(f"{'='*60}")
    
    # Emotion
    print(f"Emotion: {data['emotion'][index]}")
    
    # Situation
    situation = " ".join(data['situation'][index])
    print(f"\nSituation:\n  {situation}")
    
    # Context (dialogue history)
    print(f"\nContext (Dialogue History):")
    for i, turn in enumerate(data['context'][index]):
        turn_str = " ".join(turn)
        role = "User" if i % 2 == 0 else "Assistant"
        print(f"  Turn {i+1} [{role}]: {turn_str}")
    
    # Target
    target = " ".join(data['target'][index])
    print(f"\nTarget Response:\n  {target}")
    print()

# Print first few samples
for i in range(3):
    print_sample(train, i)

Sample #0
Emotion: sentimental

Situation:
  i remember going to the fireworks with my best friend . there was a lot of people , but it only felt like us in the world .

Context (Dialogue History):
  Turn 1 [User]: i remember going to see the fireworks with my best friend . it was the first time we ever spent time alone together . although there was a lot of people , we felt like the only people in the world .

Target Response:
  was this a friend you were in love with , or just a best friend ?

Sample #1
Emotion: sentimental

Situation:
  i remember going to the fireworks with my best friend . there was a lot of people , but it only felt like us in the world .

Context (Dialogue History):
  Turn 1 [User]: i remember going to see the fireworks with my best friend . it was the first time we ever spent time alone together . although there was a lot of people , we felt like the only people in the world .
  Turn 2 [Assistant]: was this a friend you were in love with , or just a best friend